# Appliance Energy Forecasting

Hourly appliance energy use for a single low-energy house (UCI dataset, Candanedo et al. 2017),
forecast 24 hours ahead with benchmarks, SARIMAX, XGBoost and Chronos.

This notebook walks through the analysis. The reusable code lives in `src/`;
`run_pipeline.py` runs everything in one command.

In [ ]:
import sys, warnings
warnings.filterwarnings("ignore")
sys.path.insert(0, "../src")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import data_prep, eda, benchmarks, sarima, features, ml_model, evaluate

TARGET = "Appliances"
TEST_STEPS = 14 * 24     # last 14 days held out
HORIZON = 24             # forecast horizon

## Part 1: data preparation and EDA

The raw data are sampled every 10 minutes. We parse the timestamp, check for missing
values and gaps in the sampling grid, then bin up to hourly means.

In [ ]:
hourly = data_prep.prepare()
y = hourly[TARGET]
hourly.head()

### Components of the series

The plots below show the full series, a two-week zoom, mean use by hour of day and by
day of week, and an additive decomposition with a 24-hour period.

In [ ]:
eda.overview_plot(y)
eda.profile_plots(y)
strength = eda.decomposition_plot(y)
print(f"daily seasonal strength: {strength:.3f}")

from IPython.display import Image, display
for f in ["series_overview.png", "profiles.png", "decomposition.png"]:
    display(Image(f"../outputs/figures/{f}"))

### Stationarity

ACF and PACF plots, then ADF and KPSS tests on the raw series and on differenced
versions. ADF tests the null of a unit root; KPSS tests the null of stationarity, so
the two together are more informative than either alone.

In [ ]:
eda.acf_pacf_plot(y, "raw")
eda.acf_pacf_plot(y.diff(), "first_diff")
eda.acf_pacf_plot(y.diff(24), "seasonal_diff")
display(Image("../outputs/figures/acf_pacf_raw.png"))

tests = eda.stationarity_tests({
    "raw": y, "first_diff": y.diff(),
    "seasonal_diff_24": y.diff(24),
    "seasonal_plus_first_diff": y.diff(24).diff()})
tests

Both tests agree the raw series is already stationary, so no differencing is
required (d = 0, D = 0). The strong 24-hour spikes in the ACF still show a daily
seasonal cycle, which the seasonal AR and MA terms will capture.

## Part 2: the forecasting problem

* target: `Appliances`, hourly mean energy use in Wh
* horizon: 24 hours, re-forecast every 24 hours from a rolling origin
* split: last 14 days (336 hours) held out; everything earlier is training data
* metrics: MAE, RMSE, MAPE and MASE (scaled by the in-sample daily seasonal naive
  error), plus bias. MASE is the headline metric because it is scale free and
  directly comparable with the seasonal benchmark.

In [ ]:
train, test = y.iloc[:-TEST_STEPS], y.iloc[-TEST_STEPS:]
print("train:", train.index.min(), "to", train.index.max(), f"({len(train)} h)")
print("test: ", test.index.min(), "to", test.index.max(), f"({len(test)} h)")
print("folds:", len(test) // HORIZON)

## Part 3: benchmark models

Mean, naive, daily seasonal naive, weekly seasonal naive and drift. Each is
re-computed at every 24-hour origin using only data available at that point.

In [ ]:
bench = {}
for start in range(0, len(test) - HORIZON + 1, HORIZON):
    window = test.index[start:start + HORIZON]
    history = y.loc[:window[0]].iloc[:-1]
    for name, values in benchmarks.all_benchmarks(history, HORIZON).items():
        bench.setdefault(name, []).append(pd.Series(values, index=window))
bench = pd.DataFrame({k: pd.concat(v) for k, v in bench.items()})
bench.head()

## Part 4: SARIMAX

The brief requires a full AIC grid over p = 0..6, d = 0..2, q = 0..6 (147 fits). Fits
that fail to converge can return a nonsensically low AIC, so they are flagged and
excluded before choosing the winner.

In [ ]:
# grid = sarima.grid_search(train)     # ~10 minutes, results are cached
grid = pd.read_csv("../outputs/metrics/sarima_grid_aic.csv")
order = sarima.best_order(grid)
print("best order:", order)
grid[grid.AIC > 20000].sort_values("AIC").head()

In [ ]:
hourly.index.freq = "h"; y.index.freq = "h"
X = hourly[[c for c in sarima.EXOG_COLS if c in hourly.columns]]

params = pd.read_csv("../outputs/metrics/sarimax_params.csv", index_col=0).iloc[:, 0].values
fit = sarima.refit_from_params(train, order, params, X.iloc[:-TEST_STEPS])
# to estimate from scratch instead:
# fit = sarima.fit_final(train, order, X.iloc[:-TEST_STEPS])
print(f"SARIMAX{order}{sarima.SEASONAL_ORDER}, AIC = {fit.aic:.0f}")

### Residual diagnostics

In [ ]:
resid = sarima.residual_diagnostics(fit)
display(Image("../outputs/figures/sarima_residuals.png"))
print(f"residual mean {resid.mean():.2f}, sd {resid.std():.2f}")

In [ ]:
sx, sx_ci = sarima.rolling_forecast(fit, y, test.index, X, HORIZON)
sx_ci.head()

## Parts 5 and 6: covariates and XGBoost

Features: indoor temperature and humidity sensors, outdoor weather, cyclically encoded
hour of day and day of week, lags of the target (1 to 168 hours) and shifted rolling
means and standard deviations.

Two variants are fitted to separate genuine forecasting from conditional forecasting:

* conditional: uses contemporaneous sensor and weather values, which would not be known
  in advance
* strict: those covariates are lagged 24 hours, so every input is available at the
  forecast origin

Within each 24-hour window the lag features are filled with the model's own predictions,
never with test actuals.

In [ ]:
preds = {}
for label, strict in [("xgboost_conditional", False), ("xgboost_strict", True)]:
    table = features.make_table(hourly, strict=strict)
    model, cols = ml_model.fit(table, TEST_STEPS)
    idx = table.index[-TEST_STEPS:]
    preds[label] = ml_model.recursive_forecast(model, cols, table, y, idx, HORIZON)
    if not strict:
        imp = ml_model.importance_plot(model, cols)
display(Image("../outputs/figures/xgb_importance.png"))
imp.tail(10)

## Part 7: Chronos foundation model

`amazon/chronos-bolt-small`, zero shot, no training on this series. Run
`python src/foundation.py` first; it saves the forecasts to
`outputs/forecasts/chronos_forecasts.csv`.

In [ ]:
from pathlib import Path
chronos_path = Path("../outputs/forecasts/chronos_forecasts.csv")
if chronos_path.exists():
    chronos = pd.read_csv(chronos_path, index_col=0, parse_dates=True)
    print(chronos.head())
else:
    chronos = None
    print("run: python src/foundation.py")

## Part 8: evaluation

In [ ]:
fdf = pd.DataFrame({"actual": test})
for c in bench.columns:
    fdf[c] = bench[c].reindex(test.index)
fdf["sarimax"] = sx.reindex(test.index)
for k, v in preds.items():
    fdf[k] = v.reindex(test.index)
if chronos is not None:
    fdf["chronos"] = chronos["chronos"].reindex(test.index)

results = evaluate.metrics_table(fdf, train)
results

In [ ]:
model_cols = [c for c in fdf.columns
              if c in ("sarimax", "xgboost_conditional", "xgboost_strict", "chronos")]
bench_cols = [c for c in bench.columns]

# headline figure: every model on one axis
evaluate.final_forecast_plot(train, fdf)

evaluate.forecast_plot(train, fdf, bench_cols, "forecasts_benchmarks.png",
                       "Benchmark forecasts, 14-day test period")
evaluate.forecast_plot(train, fdf, model_cols, "forecasts_models.png",
                       "Model forecasts, 14-day test period")
evaluate.forecast_plot(train, fdf.iloc[:HORIZON * 3], model_cols, "forecasts_zoom.png",
                       "First three forecast days", ci=sx_ci.iloc[:HORIZON * 3])
evaluate.error_diagnostics(fdf[["actual"] + model_cols])

for f in ["forecasts_all_models.png", "forecasts_benchmarks.png",
          "forecasts_models.png", "forecasts_zoom.png",
          "error_diagnostics.png"]:
    display(Image(f"../outputs/figures/{f}"))